# Content-Based Recommendation

In [1]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load dataset và xem thông tin tổng quan
df = pd.read_csv('../../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nPhân bố theo nguồn:")
print(df['source'].value_counts())
df.head()

Dataset shape: (10263, 13)

Phân bố theo nguồn:
source
dienmayxanh    8993
vnexpress       737
vncooking       533
Name: count, dtype: int64


,title,type_of_food,link,description,ingredients,ingredients_normalized,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...","{'tro bếp hoặc nước vo gọa', 'đường', 'cà rốt ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...","{'muối', 'mỡ lợn hoặc dầu ăn', 'đường', 'gia v...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","{'muối', 'móng giò lợn', 'nước vo gạo ngâm măn...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...","{'bộ lòng mề gà', 'muối', 'mỡ lợn', 'gia vị: m...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","{'muối', 'gừng để sơ chế bì', 'bì lợn', 'hành ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


## 1. Data Preprocessing

In [3]:
# Định nghĩa các hàm xử lý và làm sạch dữ liệu
def parse_list_string(s):
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def parse_set_string(s):
    """Parse chuỗi dạng set thành Python list (dùng cho ingredients_normalized)"""
    if pd.isna(s) or s == 'set()':
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (set, list)):
            return list(parsed)
        return []
    except:
        return []

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan

In [4]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['ingredients_normalized'] = data['ingredients_normalized'].fillna('set()')  # Thêm xử lý cho ingredients_normalized
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'ingredients_normalized', 'type_of_food', 'calories', 'cook_time']].isnull().sum())

Missing values after handling:
title                        0
description                  0
step                         0
ingredients                  0
ingredients_normalized       0
type_of_food                 0
calories                  9826
cook_time                  295
dtype: int64


In [5]:
# Áp dụng các hàm preprocessing lên dữ liệu
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

# QUAN TRỌNG: Sử dụng ingredients_normalized
data['ingredients_normalized_list'] = data['ingredients_normalized'].apply(parse_set_string)

data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

# Sử dụng ingredients_normalized_list thay vì ingredients_list
data['ingredients_clean'] = data['ingredients_normalized_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Preprocessing completed!")
print(f"Sample - Title: {data['title'].iloc[0]}")
print(f"Sample - Ingredients (normalized): {data['ingredients_normalized_list'].iloc[0][:3]}")
data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()

Preprocessing completed!
Sample - Title: Cách muối dưa hành truyền thống
Sample - Ingredients (normalized): ['tro bếp hoặc nước vo gọa', 'cà rốt trang trí tùy chọn', 'đường']


,title,title_clean,cook_time,cook_time_minutes,calories,calories_numeric
0,Cách muối dưa hành truyền thống,cách muối dưa hành truyền thống,45 phút,45.0,459 kcal,459.0
1,Su hào xào mực - món cổ Tết Bát Tràng,su hào xào mực món cổ tết bát tràng,50 phút,50.0,1.162 kcal,1162.0
2,Canh măng ngày Tết cổ truyền Hà Nội,canh măng ngày tết cổ truyền hà nội,100 phút,100.0,4.930 kcal,4930.0
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,giả hạnh nhân món ngon tết xưa hà nội,60 phút,60.0,1.112 kcal,1112.0
4,Chả bì ớt xiêm xanh,chả bì ớt xiêm xanh,60 phút,60.0,2.512 kcal,2512.0


In [6]:
# Kiểm tra kết quả preprocessing (sử dụng ingredients_normalized)
print("Sample ingredients_clean (từ ingredients_normalized):")
for i in range(min(3, len(data))):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients (normalized): {data['ingredients_normalized_list'].iloc[i][:5]}")  # Hiển thị 5 nguyên liệu đầu
    print(f"Ingredients clean text: {data['ingredients_clean'].iloc[i][:100]}...")  # Hiển thị 100 ký tự đầu

Sample ingredients_clean (từ ingredients_normalized):

Recipe 1: Cách muối dưa hành truyền thống
Ingredients (normalized): ['tro bếp hoặc nước vo gọa', 'cà rốt trang trí tùy chọn', 'đường', 'lọ sạch', 'muối hạt']
Ingredients clean text: tro bếp hoặc nước vo gọa cà rốt trang trí tùy chọn đường lọ sạch muối hạt hành củ tươi...

Recipe 2: Su hào xào mực - món cổ Tết Bát Tràng
Ingredients (normalized): ['mực khô', 'gừng', 'su hào non', 'hạt tiêu', 'đường']
Ingredients clean text: mực khô gừng su hào non hạt tiêu đường củ cà rốt rượu trắng rau mùi trang trí muối gia vị mắm mỡ lợn...

Recipe 3: Canh măng ngày Tết cổ truyền Hà Nội
Ingredients (normalized): ['hành khô', 'hành củ', 'nước dùng gà hoặc ninh xương lợn', 'nước vo gạo ngâm măng', 'gia vị: nước mắm truyền thống']
Ingredients clean text: hành khô hành củ nước dùng gà hoặc ninh xương lợn nước vo gạo ngâm măng gia vị nước mắm truyền thống...


## 2. TF-IDF Based Recommendation

Tính toán similarity dựa trên nội dung văn bản (title, description, steps) sử dụng TF-IDF và cosine similarity.

In [7]:
# Tạo TF-IDF matrix từ text features
# Kết hợp: title + description + steps + ingredients (đã chuẩn hóa)
data['combined_text'] = (
    data['title_clean'] + ' ' + 
    data['description_clean'] + ' ' + 
    data['step_clean'] + ' ' + 
    data['ingredients_clean']  # Sử dụng ingredients đã được chuẩn hóa
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

TF-IDF Matrix shape: (10263, 5000)
Vocabulary size: 5000


In [8]:
# Tính cosine similarity matrix từ TF-IDF
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")

TF-IDF Similarity Matrix shape: (10263, 10263)


In [9]:
# Hàm lấy recommendations dựa trên TF-IDF
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['tfidf_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}")
    print("\nTF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)


In [10]:
import pickle
import os

save_dir = '../Saved_models/TFIDF'
os.makedirs(save_dir, exist_ok=True)

# 1. Save TF-IDF similarity matrix
np.save(os.path.join(save_dir, 'tfidf_similarity.npy'), tfidf_similarity)

# 2. Save processed dataframe (cần cho evaluation)
data[['title', 'type_of_food', 'calories', 'cook_time', 'ingredients_normalized_list']].to_pickle(
    os.path.join(save_dir, 'tfidf_processed_data.pkl')
)

# 3. Save TF-IDF vectorizer (nếu cần tái sử dụng)
with open(os.path.join(save_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print(f"Location: {os.path.abspath(save_dir)}")
    

Location: e:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\Saved_models\TFIDF


## 3. Ingredient TF-IDF Based Recommendation

**Ingredient TF-IDF** xử lý ingredient list như text documents, tốt vì:
- Xử lý được variations trong cách viết (thịt bò, bò, beef...)
- Gán trọng số cho ingredients based on importance
- Không bị ảnh hưởng bởi exact string matching

**Ý tưởng**: Mỗi recipe là 1 document, ingredients là words. Áp dụng TF-IDF để tính similarity.

**Lưu ý**: Sử dụng **ingredients_normalized** (đã chuẩn hóa, loại bỏ số lượng).

In [11]:
# Chuẩn bị ingredient text cho TF-IDF
# Sử dụng ingredients_normalized_list (đã chuẩn hóa)
data['ingredients_text'] = data['ingredients_normalized_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Sample ingredient texts (from ingredients_normalized):")
for i in range(3):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Ingredients text: {data['ingredients_text'].iloc[i][:100]}...")

# Build TF-IDF vectorizer cho ingredients
print("\nBuilding Ingredient TF-IDF matrix...")
ingredient_tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,  # Fewer features than text-based
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,
    max_df=0.8
)

ingredient_tfidf_matrix = ingredient_tfidf_vectorizer.fit_transform(data['ingredients_text'])
print(f"Ingredient TF-IDF Matrix shape: {ingredient_tfidf_matrix.shape}")
print(f"Vocabulary size: {len(ingredient_tfidf_vectorizer.vocabulary_)}")

# Tính cosine similarity
ingredient_tfidf_similarity = cosine_similarity(ingredient_tfidf_matrix, ingredient_tfidf_matrix)
print(f"Ingredient TF-IDF Similarity Matrix shape: {ingredient_tfidf_similarity.shape}")

Sample ingredient texts (from ingredients_normalized):

Cách muối dưa hành truyền thống
   Ingredients text: tro bếp hoặc nước vo gọa cà rốt trang trí tùy chọn đường lọ sạch muối hạt hành củ tươi...

Su hào xào mực - món cổ Tết Bát Tràng
   Ingredients text: mực khô gừng su hào non hạt tiêu đường củ cà rốt rượu trắng rau mùi trang trí muối gia vị mắm mỡ lợn...

Canh măng ngày Tết cổ truyền Hà Nội
   Ingredients text: hành khô hành củ nước dùng gà hoặc ninh xương lợn nước vo gạo ngâm măng gia vị nước mắm truyền thống...

Building Ingredient TF-IDF matrix...
Ingredient TF-IDF Matrix shape: (10263, 2000)
Vocabulary size: 2000
Ingredient TF-IDF Similarity Matrix shape: (10263, 10263)


In [12]:
# Hàm lấy recommendations dựa trên Ingredient TF-IDF
def get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['ing_tfidf_score'] = scores
    
    return result

def recommend_by_title_ingredient_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients (normalized): {list(df.loc[recipe_idx, 'ingredients_normalized_list'])[:5]}...")
    print("\nIngredient TF-IDF Recommendations:")
    
    return get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [13]:
# Save các file quan trọng cho Ingredient TF-IDF evaluation
save_dir = '../Saved_models/Ingredient_TFIDF'
os.makedirs(save_dir, exist_ok=True)

# 1. Save Ingredient TF-IDF similarity matrix
np.save(os.path.join(save_dir, 'ingredient_tfidf_similarity.npy'), ingredient_tfidf_similarity)

# 2. Save Ingredient TF-IDF vectorizer (cần cho inference)
with open(os.path.join(save_dir, 'ingredient_tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(ingredient_tfidf_vectorizer, f)

print("\nSaved")


Saved


## 4. Hybrid Approach

Kết hợp 2 phương pháp với trọng số: TF-IDF (0.4) + Ingredient TF-IDF (0.6)

**Rationale**: Ingredients quan trọng nhất trong food recommendation, nên tăng Ingredient TF-IDF weight

**Lưu ý**: Cả hai phương pháp đều sử dụng **ingredients_normalized** làm cơ sở.

In [14]:
# Hàm tính hybrid similarity (kết hợp TF-IDF và Ingredient TF-IDF)
def compute_hybrid_similarity(tfidf_sim, ing_tfidf_sim, 
                              w_tfidf=0.4, w_ing_tfidf=0.6):
    total_weight = w_tfidf + w_ing_tfidf
    w_tfidf /= total_weight
    w_ing_tfidf /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Ingredient TF-IDF={w_ing_tfidf:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_ing_tfidf * ing_tfidf_sim)
    
    return hybrid_similarity

In [15]:
# Tính hybrid similarity matrix
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    ingredient_tfidf_similarity,
    w_tfidf=0.4,
    w_ing_tfidf=0.6
)
print(f"Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")

Weights: TF-IDF=0.40, Ingredient TF-IDF=0.60
Hybrid Similarity Matrix shape: (10263, 10263)


In [16]:
# Hàm lấy recommendations dựa trên hybrid approach
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['ing_tfidf_score'] = [ing_tfidf_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, ing_tfidf_sim, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print(f"   Ingredients (normalized): {list(df.loc[recipe_idx, 'ingredients_normalized_list'])[:5]}...")
    print("\nHybrid Recommendations:")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n)

In [17]:
# Save Hybrid similarity matrix cho evaluation
save_dir = '../Saved_models/Hybrid'
os.makedirs(save_dir, exist_ok=True)

# Chỉ save hybrid similarity matrix (đã bao gồm cả TF-IDF và Ingredient TF-IDF)
np.save(os.path.join(save_dir, 'hybrid_similarity.npy'), hybrid_similarity_matrix)

print("\nSaved!")


Saved!


## 5. Keyword-Based Recommendation

Extract keywords chính từ title và ingredients_normalized (nguyên liệu chính).

**Ưu điểm**: Đơn giản, focus vào main keywords, dễ giải thích

**Lưu ý**: Keywords được trích xuất từ title và **ingredients_normalized**

In [18]:
# Hàm lấy recommendations dựa trên keywords
def get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['keyword_score'] = scores
    
    return result

def recommend_by_title_keyword(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Keywords (từ title + ingredients_normalized): {list(df.loc[recipe_idx, 'keywords'])[:10]}...")
    print("\nKeyword-Based Recommendations:")
    
    return get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [19]:
# Hàm extract keywords/tags từ title và ingredients_normalized
def extract_keywords(title, ingredients_normalized_list):
    """
    Extract main keywords from recipe title and normalized ingredients
    Focus on: ingredients, cooking methods, dish types
    """
    keywords = set()
    
    if pd.notna(title):
        title = clean_text(title)
        
        # Common Vietnamese cooking methods and dish types
        cooking_methods = ['xào', 'nướng', 'luộc', 'chiên', 'hấp', 'kho', 'rim', 'rang', 
                           'canh', 'súp', 'cháo', 'gỏi', 'nộm', 'salad', 'bún', 'phở', 
                           'mì', 'cơm', 'bánh', 'chè', 'sinh tố']
        
        # Add cooking methods from title
        for method in cooking_methods:
            if method in title:
                keywords.add(method)
        
        # Add significant words from title (length > 2)
        for word in title.split():
            if len(word) > 2:
                keywords.add(word)
    
    # QUAN TRỌNG: Thêm keywords từ ingredients_normalized
    if ingredients_normalized_list:
        for ingredient in ingredients_normalized_list:
            cleaned_ing = clean_text(str(ingredient))
            if len(cleaned_ing) > 2:
                keywords.add(cleaned_ing)
    
    return keywords

In [20]:
# Apply keyword extraction to all recipes (sử dụng ingredients_normalized_list)
data['keywords'] = data.apply(
    lambda row: extract_keywords(row['title'], row['ingredients_normalized_list']), 
    axis=1
)
print(f"Keyword extraction completed!")

# Show examples
print("\nSample keywords (từ title + ingredients_normalized):")
for i in range(5):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Keywords: {list(data['keywords'].iloc[i])[:10]}...")  # Hiển thị 10 keywords đầu

Keyword extraction completed!

Sample keywords (từ title + ingredients_normalized):

Cách muối dưa hành truyền thống
   Keywords: ['truyền', 'thống', 'tro bếp hoặc nước vo gọa', 'hành', 'cà rốt trang trí tùy chọn', 'muối hạt', 'đường', 'cách', 'lọ sạch', 'muối']...

Su hào xào mực - món cổ Tết Bát Tràng
   Keywords: ['mực khô', 'gừng', 'su hào non', 'xào', 'món', 'gia vị mắm', 'mỡ lợn hoặc dầu ăn', 'hạt tiêu', 'đường', 'củ cà rốt']...

Canh măng ngày Tết cổ truyền Hà Nội
   Keywords: ['hành khô', 'canh', 'truyền', 'hành củ', 'nước dùng gà hoặc ninh xương lợn', 'nước vo gạo ngâm măng', 'ngày', 'măng khô', 'móng giò lợn', 'gia vị nước mắm truyền thống']...

Giả hạnh nhân - món ngon Tết xưa Hà Nội
   Keywords: ['ngon', 'lạc', 'hạnh', 'củ đậu', 'mỡ lợn', 'gia vị mắm', 'nội', 'hành khô', 'hạt đậu hà lan', 'nhân']...

Chả bì ớt xiêm xanh
   Keywords: ['chả', 'hành khô', 'chanh', 'gia vị mắm cốt', 'ớt xiêm xanh', 'xiêm', 'giò sống', 'xanh', 'lá chuối gói', 'gừng để sơ chế bì']...


In [21]:
# Helper functions cho Keyword similarity (sử dụng Jaccard cho keyword sets)
def jaccard_similarity(set1, set2):
    """
    Tính Jaccard similarity giữa 2 sets
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(items_list):
    """
    Tính Jaccard similarity matrix cho list of sets
    Sử dụng cho keyword-based recommendation
    """
    n = len(items_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(items_list[i], items_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix

print("Jaccard helper functions defined (for keyword similarity only)!")

Jaccard helper functions defined (for keyword similarity only)!


In [22]:
# Tính Keyword-based similarity matrix
keywords_sets = data['keywords'].tolist()
keyword_similarity_matrix = compute_jaccard_similarity_matrix(keywords_sets)
print(f"Keyword Similarity Matrix shape: {keyword_similarity_matrix.shape}")

Keyword Similarity Matrix shape: (10263, 10263)


In [23]:
# Save Keyword-based similarity matrix cho evaluation
save_dir = '../Saved_models/Keyword'
os.makedirs(save_dir, exist_ok=True)

# Chỉ save keyword similarity matrix
np.save(os.path.join(save_dir, 'keyword_similarity.npy'), keyword_similarity_matrix)

print(f"\nSaved")


Saved
